# FACED hierarchical Frequency–Band–Channel–Time Transformer

## Purpose

This notebook keeps the source-isolated FACED data protocol and the frequency-resolved
`sqrt-JSD` feature construction from the earlier notebook, but replaces the legacy
`flatten(channel × frequency) → temporal Transformer` backbone with a hierarchical model:

```text
46 native frequency bins
    → five independent frequency-band encoders
    → within-channel band Transformer
    → cross-channel Transformer
    → temporal Transformer
    → attentive mean + standard-deviation pooling
    → nine-class classifier
```

The main structural changes are:

- the 46 native frequency points are split according to their physical bands instead of being
  treated as an undifferentiated vector;
- each channel first learns a five-band representation;
- channel tokens remain explicit during spatial modeling and are not compressed back to the
  original feature dimension;
- a global spatial token summarizes all channels at each time step;
- temporal pooling preserves both the weighted mean state and temporal variability;
- an optional channel-voting auxiliary head encourages individual channels to remain
  discriminative;
- an optional gradient-reversal subject head can be enabled after the architecture-only
  baseline is established.

The data preparation, source-fit/source-development split, target locking, sanity gate,
checkpoint policy and final target evaluation follow the first notebook.

## Recommended experiment order

1. Run the default `base` preset with `USE_SUBJECT_ADVERSARY=False`.
2. Compare source-development Macro-F1 against the legacy notebook.
3. Turn on `USE_SUBJECT_ADVERSARY=True` only after the architecture-only result is recorded.
4. Try the `large` preset only when the base model is stable and GPU memory allows it.


## Parameters

所有需要调整的参数集中在这里。每次材料性修改都使用新的 `RUN_NAME`。

In [1]:
# ------------------------------ Protocol ------------------------------
RUN_NAME = "faced_hierarchical_fbct_base_seed42"
FOLD = 10
SEED = 42
SOURCE_DEV_SUBJECTS = 37
RUN_TRAINING = True
EVALUATE_TARGET_AFTER_LOCK = False
TARGET_CHECKPOINT = "best_source_dev"  # "best_source_dev" or "final"

# ------------------------------ Feature -------------------------------
EPSILON = 1e-12
FEATURE_STORAGE_DTYPE = "float16"       # PSD/JSD materialization cache
STANDARDIZE_SOURCE_FEATURES = True
STANDARDIZED_TENSOR_DTYPE = "float32"   # store standardized trials; AMP handles compute dtype
STANDARDIZE_CHUNK_TRIALS = 32

# ------------------------------ Model --------------------------------
# "base" is the recommended first run. "large" is deliberately more expensive.
MODEL_PRESET = "base"

MODEL_PRESETS = {
    "base": {
        "frequency_hidden": 64,
        "band_d_model": 64,
        "band_heads": 4,
        "band_layers": 1,
        "channel_d_model": 256,
        "channel_heads": 8,
        "channel_layers": 2,
        "temporal_d_model": 512,
        "temporal_heads": 8,
        "temporal_layers": 4,
        "dropout": 0.15,
        "batch_size": 64,
    },
    "large": {
        "frequency_hidden": 96,
        "band_d_model": 96,
        "band_heads": 6,
        "band_layers": 2,
        "channel_d_model": 384,
        "channel_heads": 6,
        "channel_layers": 3,
        "temporal_d_model": 640,
        "temporal_heads": 10,
        "temporal_layers": 6,
        "dropout": 0.18,
        "batch_size": 32,
    },
}
if MODEL_PRESET not in MODEL_PRESETS:
    raise ValueError(f"Unknown MODEL_PRESET={MODEL_PRESET!r}")
MODEL_CFG = MODEL_PRESETS[MODEL_PRESET]

FREQUENCY_HIDDEN = MODEL_CFG["frequency_hidden"]
BAND_D_MODEL = MODEL_CFG["band_d_model"]
BAND_HEADS = MODEL_CFG["band_heads"]
BAND_LAYERS = MODEL_CFG["band_layers"]
CHANNEL_D_MODEL = MODEL_CFG["channel_d_model"]
CHANNEL_HEADS = MODEL_CFG["channel_heads"]
CHANNEL_LAYERS = MODEL_CFG["channel_layers"]
TEMPORAL_D_MODEL = MODEL_CFG["temporal_d_model"]
TEMPORAL_HEADS = MODEL_CFG["temporal_heads"]
TEMPORAL_LAYERS = MODEL_CFG["temporal_layers"]
DROPOUT = MODEL_CFG["dropout"]

FFN_RATIO = 4
USE_CHANNEL_VOTE = True
CHANNEL_VOTE_LOSS_WEIGHT = 0.15

# Subject adversarial learning is intentionally disabled for the first architecture-only run.
USE_SUBJECT_ADVERSARY = False
SUBJECT_ADV_WEIGHT = 0.05
SUBJECT_ADV_WARMUP_EPOCHS = 10

# ----------------------------- Training -------------------------------
EPOCHS = 80
BATCH_SIZE = MODEL_CFG["batch_size"]
EVAL_BATCH_SIZE = 192
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-2
MAX_GRAD_NORM = 1.0
LABEL_SMOOTHING = 0.05
WARMUP_EPOCHS = 5
MIN_LR_RATIO = 0.05
EVAL_EVERY = 1
EARLY_STOPPING_PATIENCE = 12

# -------------------------- Runtime efficiency ------------------------
DEVICE = "auto"
USE_AMP = True
AMP_DTYPE = "bfloat16"  # recommended on recent NVIDIA GPUs; use "float16" if unsupported
CACHE_DATA_ON_DEVICE = True
COMPILE_MODEL = False   # turn on only after the ordinary model runs correctly

# -------------------------- Hard sanity gate --------------------------
RUN_SANITY_GATE = True
SANITY_SAMPLES_PER_CLASS = 2
SANITY_MAX_STEPS = 300
SANITY_LEARNING_RATE = 1e-4  # validated in float32; higher values collapse
SANITY_TARGET_ACCURACY = 0.99
SANITY_TARGET_LOSS = 0.05


## Setup

### 1. Imports, paths, deterministic runtime

In [2]:
from __future__ import annotations

import contextlib
import csv
import gc
import hashlib
import inspect
import json
import math
import random
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src" / "cmrd" / "faced.py").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the CMRD repository")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def configure_runtime() -> None:
    # Exact reproducibility is useful for comparisons, while TF32 still accelerates
    # large matrix multiplications on recent NVIDIA GPUs.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision("high")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from cmrd.faced import (
    EEG_CHANNEL_NAMES,
    EMOTION_NAMES,
    SUBJECTS,
    VIDEO_LABELS,
    VIDEOS,
    official_fold_subjects,
)
from cmrd.faced_psd_jsd_experiment import (
    SpectraStore,
    fit_reference,
    materialize_split,
)

seed_everything(SEED)
configure_runtime()

device = torch.device(
    "cuda" if DEVICE == "auto" and torch.cuda.is_available()
    else DEVICE if DEVICE != "auto" else "cpu"
)

if AMP_DTYPE == "bfloat16":
    amp_dtype = torch.bfloat16
elif AMP_DTYPE == "float16":
    amp_dtype = torch.float16
else:
    raise ValueError("AMP_DTYPE must be 'bfloat16' or 'float16'")

amp_enabled = bool(USE_AMP and device.type == "cuda")


def autocast_context():
    if amp_enabled:
        return torch.autocast(device_type="cuda", dtype=amp_dtype)
    return contextlib.nullcontext()


RUN_ROOT = REPO_ROOT / "runs" / RUN_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO_ROOT)
print("Device:", device)
print("AMP:", amp_enabled, AMP_DTYPE if amp_enabled else "disabled")
print("Model preset:", MODEL_PRESET)
print("Output:", RUN_ROOT)


Repository: C:\Users\Lin\Documents\Arbitruam\CMRD-Cute-Mew-Really-Delighting
Device: cuda
AMP: True bfloat16
Model preset: base
Output: C:\Users\Lin\Documents\Arbitruam\CMRD-Cute-Mew-Really-Delighting\runs\faced_hierarchical_fbct_base_seed42


## Data

### 2. Locate PSD cache and freeze source/dev/target subjects

In [3]:
BASE_CACHE = REPO_ROOT / "runs" / "faced_native_compact_base_seed42" / "cache" / "native_spectra"
manifest_paths = sorted(BASE_CACHE.glob("*/manifest.json"))
complete = []
for manifest_path in manifest_paths:
    payload = json.loads(manifest_path.read_text(encoding="utf-8"))
    if payload.get("all_subjects_complete") and len(payload.get("subjects_complete", [])) == SUBJECTS:
        complete.append((manifest_path, payload))
if len(complete) != 1:
    raise RuntimeError(f"Expected exactly one complete native PSD cache, found {len(complete)}")

SPECTRA_MANIFEST_PATH, spectra_manifest = complete[0]
SPECTRA_ROOT = SPECTRA_MANIFEST_PATH.parent
BAND_NAMES = tuple(spectra_manifest["band_names"])
BAND_SIZES = tuple(map(int, spectra_manifest["band_sizes"]))
NATIVE_FEATURES = sum(BAND_SIZES)  # 3+4+6+16+17 = 46

source_subjects, outer_target_subjects = official_fold_subjects(FOLD)
shuffled_source = np.random.default_rng(SEED).permutation(source_subjects)
dev_subjects = tuple(sorted(map(int, shuffled_source[:SOURCE_DEV_SUBJECTS])))
fit_subjects = tuple(sorted(map(int, shuffled_source[SOURCE_DEV_SUBJECTS:])))
outer_target_subjects = tuple(map(int, outer_target_subjects))

assert len(fit_subjects) == 74 and len(dev_subjects) == 37
assert set(fit_subjects).isdisjoint(dev_subjects)
assert (set(fit_subjects) | set(dev_subjects)).isdisjoint(outer_target_subjects)

display(pd.DataFrame({
    "role": ["source fit", "source validation", "outer target (locked)"],
    "subjects": [len(fit_subjects), len(dev_subjects), len(outer_target_subjects)],
    "trials": [len(fit_subjects) * VIDEOS, len(dev_subjects) * VIDEOS, len(outer_target_subjects) * VIDEOS],
}))
display(pd.DataFrame({"band": BAND_NAMES, "native_frequency_points": BAND_SIZES}))
print("Compact model input per window:", (len(EEG_CHANNEL_NAMES), NATIVE_FEATURES))

,role,subjects,trials
0,source fit,74,2072
1,source validation,37,1036
2,outer target (locked),12,336


,band,native_frequency_points
0,delta,3
1,theta,4
2,alpha,6
3,beta,16
4,gamma,17


Compact model input per window: (30, 46)


### 3. Fit source-only reference and build frequency-resolved sqrt-JSD

In [4]:
def compact_native_bins(padded: np.ndarray) -> np.ndarray:
    # [N,T,C,5,17] -> [N,T,C,46], dropping only deterministic padding.
    return np.concatenate(
        [padded[..., band, :size] for band, size in enumerate(BAND_SIZES)],
        axis=-1,
    )


def fit_compact_standardizer(values: np.ndarray, chunk_trials: int = 32):
    total = np.zeros(values.shape[2:], dtype=np.float64)
    square = np.zeros_like(total)
    count = 0
    for start in range(0, len(values), chunk_trials):
        chunk = values[start:start + chunk_trials].astype(np.float32)
        total += chunk.sum(axis=(0, 1), dtype=np.float64)
        square += np.square(chunk).sum(axis=(0, 1), dtype=np.float64)
        count += chunk.shape[0] * chunk.shape[1]
    mean = total / count
    variance = np.maximum(square / count - np.square(mean), 0.0)
    std = np.sqrt(variance)
    std[std < 1e-7] = 1.0
    return mean.astype(np.float32), std.astype(np.float32)


store = SpectraStore(SPECTRA_ROOT)
reference, reference_windows = fit_reference(store, fit_subjects)

train_raw = materialize_split(store, fit_subjects, reference, EPSILON, FEATURE_STORAGE_DTYPE)
train_features = compact_native_bins(train_raw.pop("x"))
train_labels = np.asarray(train_raw["y"], dtype=np.int64)
train_subject_ids = np.asarray(train_raw["subjects"], dtype=np.int64)
train_jsd_error = float(train_raw["maximum_invariant_error"])
del train_raw
gc.collect()

dev_raw = materialize_split(store, dev_subjects, reference, EPSILON, FEATURE_STORAGE_DTYPE)
dev_features = compact_native_bins(dev_raw.pop("x"))
dev_labels = np.asarray(dev_raw["y"], dtype=np.int64)
dev_subject_ids = np.asarray(dev_raw["subjects"], dtype=np.int64)
dev_jsd_error = float(dev_raw["maximum_invariant_error"])
del dev_raw
gc.collect()

feature_mean, feature_std = fit_compact_standardizer(train_features)
if not STANDARDIZE_SOURCE_FEATURES:
    feature_mean = np.zeros_like(feature_mean)
    feature_std = np.ones_like(feature_std)

target_overlap = set(store.loaded_subjects) & set(outer_target_subjects)
if target_overlap:
    raise RuntimeError(f"Outer target was loaded during source preparation: {sorted(target_overlap)}")

audit = {
    "feature": "frequency_resolved_sqrt_jsd",
    "sqrt_invariant": "sum_frequency(field**2) == JSD",
    "band_sizes": list(BAND_SIZES),
    "compact_feature_count": NATIVE_FEATURES,
    "reference_scope": "source_fit_subjects_only",
    "reference_windows": reference_windows,
    "fit_subjects": list(fit_subjects),
    "development_subjects": list(dev_subjects),
    "outer_target_subjects": list(outer_target_subjects),
    "loaded_subjects": sorted(store.loaded_subjects),
    "target_loaded": False,
    "maximum_sqrt_jsd_reconstruction_error": max(train_jsd_error, dev_jsd_error),
}
(RUN_ROOT / "source_isolation_audit.json").write_text(
    json.dumps(audit, indent=2, ensure_ascii=False), encoding="utf-8"
)

print("Train shape:", train_features.shape)
print("Dev shape:", dev_features.shape)
print("Max sqrt-JSD reconstruction error:", audit["maximum_sqrt_jsd_reconstruction_error"])
print("Outer target loaded:", audit["target_loaded"])

Train shape: (2072, 30, 30, 46)
Dev shape: (1036, 30, 30, 46)
Max sqrt-JSD reconstruction error: 5.960464477539063e-08
Outer target loaded: False


### 4. Standardize once and cache tensor splits

The legacy notebook converted `float16 → float32` and standardized every sample in every
epoch. This notebook standardizes each trial once, converts it to a contiguous tensor and,
by default, keeps the train/development tensors on the GPU. The complete source data occupies
well below 1 GB in `float32`, so this removes the DataLoader and NumPy preprocessing bottleneck
on a 16 GB GPU.

Set `CACHE_DATA_ON_DEVICE=False` only when GPU memory is constrained.


In [5]:
def resolve_storage_dtype(name: str) -> torch.dtype:
    table = {
        "float32": torch.float32,
        "float16": torch.float16,
        "bfloat16": torch.bfloat16,
    }
    if name not in table:
        raise ValueError(f"Unsupported STANDARDIZED_TENSOR_DTYPE={name!r}")
    return table[name]


def standardize_to_tensor(
    values: np.ndarray,
    mean: np.ndarray,
    std: np.ndarray,
    chunk_trials: int,
    dtype: torch.dtype,
) -> torch.Tensor:
    output = torch.empty(values.shape, dtype=dtype)
    for start in range(0, len(values), chunk_trials):
        end = min(start + chunk_trials, len(values))
        chunk = values[start:end].astype(np.float32)
        chunk = (chunk - mean) / std
        output[start:end].copy_(
            torch.from_numpy(np.ascontiguousarray(chunk)).to(dtype=dtype)
        )
    return output


@dataclass
class TensorSplit:
    x: torch.Tensor
    y: torch.Tensor
    subject_id: torch.Tensor
    subject_class: torch.Tensor

    def __len__(self) -> int:
        return int(self.y.numel())


def make_tensor_split(
    features: np.ndarray,
    labels: np.ndarray,
    subject_ids: np.ndarray,
    mean: np.ndarray,
    std: np.ndarray,
    subject_to_class: Optional[dict[int, int]] = None,
) -> TensorSplit:
    storage_dtype = resolve_storage_dtype(STANDARDIZED_TENSOR_DTYPE)
    x = standardize_to_tensor(
        features, mean, std, STANDARDIZE_CHUNK_TRIALS, storage_dtype
    )
    y = torch.as_tensor(labels, dtype=torch.long)
    subject_id = torch.as_tensor(subject_ids, dtype=torch.long)
    if subject_to_class is None:
        subject_class = torch.full_like(subject_id, -1)
    else:
        mapped = [subject_to_class[int(s)] for s in subject_ids]
        subject_class = torch.as_tensor(mapped, dtype=torch.long)

    data_device = device if CACHE_DATA_ON_DEVICE and device.type == "cuda" else torch.device("cpu")
    return TensorSplit(
        x=x.to(data_device),
        y=y.to(data_device),
        subject_id=subject_id.to(data_device),
        subject_class=subject_class.to(data_device),
    )


def iterate_batches(
    split: TensorSplit,
    batch_size: int,
    shuffle: bool,
):
    n = len(split)
    index_device = split.y.device
    indices = (
        torch.randperm(n, device=index_device)
        if shuffle else
        torch.arange(n, device=index_device)
    )
    for start in range(0, n, batch_size):
        index = indices[start:start + batch_size]
        yield (
            split.x.index_select(0, index),
            split.y.index_select(0, index),
            split.subject_id.index_select(0, index),
            split.subject_class.index_select(0, index),
        )


fit_subject_to_class = {int(subject): i for i, subject in enumerate(fit_subjects)}

train_split = make_tensor_split(
    train_features,
    train_labels,
    train_subject_ids,
    feature_mean,
    feature_std,
    fit_subject_to_class,
)
dev_split = make_tensor_split(
    dev_features,
    dev_labels,
    dev_subject_ids,
    feature_mean,
    feature_std,
    None,
)

del train_features, dev_features
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

sample_x = train_split.x[0]
sample_y = train_split.y[0]
sample_subject = train_split.subject_id[0]

def tensor_megabytes(tensor: torch.Tensor) -> float:
    return tensor.numel() * tensor.element_size() / (1024 ** 2)

display(pd.DataFrame({
    "split": ["train", "development"],
    "trials": [len(train_split), len(dev_split)],
    "tensor_device": [str(train_split.x.device), str(dev_split.x.device)],
    "feature_memory_MB": [
        tensor_megabytes(train_split.x),
        tensor_megabytes(dev_split.x),
    ],
}))
print("One trial:", tuple(sample_x.shape), "label:", int(sample_y), "subject:", int(sample_subject))


,split,trials,tensor_device,feature_memory_MB
0,train,2072,cuda:0,327.227783
1,development,1036,cuda:0,163.613892


One trial: (30, 30, 46) label: 0 subject: 13


## Model

### 5. Hierarchical Frequency–Band–Channel–Time Transformer

The model has four explicit stages:

1. **Frequency encoders:** one MLP per physical frequency band.
2. **Band Transformer:** five band tokens interact within each channel and time step.
3. **Channel Transformer:** all channel tokens interact while remaining explicit.
4. **Temporal Transformer:** spatial summaries interact across the 30 time windows.

The final attentive-statistics pooling concatenates a weighted temporal mean and weighted
temporal standard deviation. The optional channel-vote head is an auxiliary classifier rather
than a replacement for the global branch.


In [6]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, ratio: int, dropout: float):
        super().__init__()
        hidden = ratio * d_model
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        return self.net(value)


class PreNormTransformerBlock(nn.Module):
    def __init__(self, d_model: int, nhead: int, dropout: float, ffn_ratio: int):
        super().__init__()
        if d_model % nhead != 0:
            raise ValueError(f"d_model={d_model} must be divisible by nhead={nhead}")
        self.norm1 = nn.LayerNorm(d_model)
        self.attention = nn.MultiheadAttention(
            d_model,
            nhead,
            dropout=dropout,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, ffn_ratio, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        value: torch.Tensor,
        key_padding_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        normalized = self.norm1(value)
        attended, _ = self.attention(
            normalized,
            normalized,
            normalized,
            key_padding_mask=key_padding_mask,
            need_weights=False,
        )
        value = value + self.dropout(attended)
        value = value + self.ffn(self.norm2(value))
        return value


class FrequencyBandEncoder(nn.Module):
    def __init__(
        self,
        band_sizes: tuple[int, ...],
        frequency_hidden: int,
        d_model: int,
        nhead: int,
        layers: int,
        dropout: float,
        ffn_ratio: int,
    ):
        super().__init__()
        self.band_sizes = tuple(int(size) for size in band_sizes)
        self.band_slices = []
        offset = 0
        for size in self.band_sizes:
            self.band_slices.append(slice(offset, offset + size))
            offset += size

        self.encoders = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(size),
                nn.Linear(size, frequency_hidden),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(frequency_hidden, d_model),
            )
            for size in self.band_sizes
        ])
        self.band_embedding = nn.Parameter(
            torch.zeros(1, 1, len(self.band_sizes), d_model)
        )
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.blocks = nn.ModuleList([
            PreNormTransformerBlock(d_model, nhead, dropout, ffn_ratio)
            for _ in range(layers)
        ])
        self.output_norm = nn.LayerNorm(d_model)
        nn.init.trunc_normal_(self.band_embedding, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        # value: [B,T,C,46]
        encoded_bands = [
            encoder(value[..., band_slice])
            for encoder, band_slice in zip(self.encoders, self.band_slices)
        ]
        tokens = torch.stack(encoded_bands, dim=-2)
        tokens = tokens + self.band_embedding

        batch, time_steps, channels, bands, d_model = tokens.shape
        tokens = tokens.reshape(batch * time_steps * channels, bands, d_model)
        cls = self.cls_token.expand(tokens.shape[0], -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)

        for block in self.blocks:
            tokens = block(tokens)
        channel_token = self.output_norm(tokens[:, 0])
        return channel_token.reshape(batch, time_steps, channels, d_model)


class ChannelEncoder(nn.Module):
    def __init__(
        self,
        channels: int,
        input_dim: int,
        d_model: int,
        nhead: int,
        layers: int,
        dropout: float,
        ffn_ratio: int,
    ):
        super().__init__()
        self.channels = channels
        self.input_projection = nn.Linear(input_dim, d_model)
        self.channel_embedding = nn.Parameter(torch.zeros(1, 1, channels, d_model))
        self.spatial_cls = nn.Parameter(torch.zeros(1, 1, d_model))
        self.blocks = nn.ModuleList([
            PreNormTransformerBlock(d_model, nhead, dropout, ffn_ratio)
            for _ in range(layers)
        ])
        self.output_norm = nn.LayerNorm(d_model)
        nn.init.trunc_normal_(self.channel_embedding, std=0.02)
        nn.init.trunc_normal_(self.spatial_cls, std=0.02)

    def forward(self, channel_tokens: torch.Tensor):
        # channel_tokens: [B,T,C,D_band]
        tokens = self.input_projection(channel_tokens) + self.channel_embedding
        batch, time_steps, channels, d_model = tokens.shape
        tokens = tokens.reshape(batch * time_steps, channels, d_model)
        cls = self.spatial_cls.expand(tokens.shape[0], -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)

        for block in self.blocks:
            tokens = block(tokens)
        tokens = self.output_norm(tokens)

        spatial_token = tokens[:, 0].reshape(batch, time_steps, d_model)
        explicit_channels = tokens[:, 1:].reshape(
            batch, time_steps, channels, d_model
        )
        return spatial_token, explicit_channels


class LearnableTemporalEncoding(nn.Module):
    def __init__(self, time_steps: int, d_model: int, dropout: float):
        super().__init__()
        self.position = nn.Parameter(torch.zeros(1, time_steps, d_model))
        self.dropout = nn.Dropout(dropout)
        nn.init.trunc_normal_(self.position, std=0.02)

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        return self.dropout(value + self.position[:, :value.shape[1]])


class AttentiveStatsPooling(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        hidden = max(d_model // 2, 64)
        self.score = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(
        self,
        value: torch.Tensor,
        valid_mask: Optional[torch.Tensor] = None,
    ):
        scores = self.score(value).squeeze(-1)
        if valid_mask is not None:
            scores = scores.masked_fill(~valid_mask, torch.finfo(scores.dtype).min)
        weights = torch.softmax(scores, dim=1)
        mean = torch.sum(weights.unsqueeze(-1) * value, dim=1)
        centered = value - mean.unsqueeze(1)
        variance = torch.sum(
            weights.unsqueeze(-1) * centered.square(),
            dim=1,
        )
        std = torch.sqrt(variance.clamp_min(1e-6))
        return torch.cat([mean, std], dim=-1), weights


class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, value: torch.Tensor, coefficient: float):
        ctx.coefficient = float(coefficient)
        return value.view_as(value)

    @staticmethod
    def backward(ctx, gradient: torch.Tensor):
        return -ctx.coefficient * gradient, None


class HierarchicalFBCT(nn.Module):
    def __init__(
        self,
        channels: int,
        band_sizes: tuple[int, ...],
        time_steps: int,
        classes: int,
        source_subject_classes: int,
        dropout: float,
        use_channel_vote: bool,
        use_subject_adversary: bool,
    ):
        super().__init__()
        self.channels = channels
        self.feature_dim = int(sum(band_sizes))
        self.classes = classes
        self.use_channel_vote = bool(use_channel_vote)
        self.use_subject_adversary = bool(use_subject_adversary)

        self.frequency_band_encoder = FrequencyBandEncoder(
            band_sizes=band_sizes,
            frequency_hidden=FREQUENCY_HIDDEN,
            d_model=BAND_D_MODEL,
            nhead=BAND_HEADS,
            layers=BAND_LAYERS,
            dropout=dropout,
            ffn_ratio=FFN_RATIO,
        )
        self.channel_encoder = ChannelEncoder(
            channels=channels,
            input_dim=BAND_D_MODEL,
            d_model=CHANNEL_D_MODEL,
            nhead=CHANNEL_HEADS,
            layers=CHANNEL_LAYERS,
            dropout=dropout,
            ffn_ratio=FFN_RATIO,
        )

        self.temporal_input = nn.Linear(CHANNEL_D_MODEL, TEMPORAL_D_MODEL)
        self.temporal_position = LearnableTemporalEncoding(
            time_steps, TEMPORAL_D_MODEL, dropout
        )
        self.temporal_blocks = nn.ModuleList([
            PreNormTransformerBlock(
                TEMPORAL_D_MODEL,
                TEMPORAL_HEADS,
                dropout,
                FFN_RATIO,
            )
            for _ in range(TEMPORAL_LAYERS)
        ])
        self.temporal_norm = nn.LayerNorm(TEMPORAL_D_MODEL)
        self.temporal_pool = AttentiveStatsPooling(TEMPORAL_D_MODEL)

        pooled_dim = 2 * TEMPORAL_D_MODEL
        self.classifier = nn.Sequential(
            nn.LayerNorm(pooled_dim),
            nn.Linear(pooled_dim, TEMPORAL_D_MODEL),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(TEMPORAL_D_MODEL, classes),
        )

        if self.use_channel_vote:
            self.channel_gate = nn.Sequential(
                nn.LayerNorm(CHANNEL_D_MODEL),
                nn.Linear(CHANNEL_D_MODEL, 1),
            )
            self.channel_classifier = nn.Sequential(
                nn.LayerNorm(CHANNEL_D_MODEL),
                nn.Linear(CHANNEL_D_MODEL, classes),
            )
        else:
            self.channel_gate = None
            self.channel_classifier = None

        if self.use_subject_adversary:
            self.subject_classifier = nn.Sequential(
                nn.LayerNorm(pooled_dim),
                nn.Linear(pooled_dim, TEMPORAL_D_MODEL),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(TEMPORAL_D_MODEL, source_subject_classes),
            )
        else:
            self.subject_classifier = None

    def forward(
        self,
        value: torch.Tensor,
        valid_mask: Optional[torch.Tensor] = None,
        subject_grl_coefficient: float = 0.0,
    ):
        if value.ndim != 4:
            raise ValueError(f"Expected [B,T,C,F], got {tuple(value.shape)}")
        if value.shape[2] != self.channels or value.shape[3] != self.feature_dim:
            raise ValueError(
                f"Expected C={self.channels}, F={self.feature_dim}; got {tuple(value.shape[2:])}"
            )
        if valid_mask is None:
            valid_mask = torch.ones(
                value.shape[:2],
                dtype=torch.bool,
                device=value.device,
            )

        channel_tokens = self.frequency_band_encoder(value)
        spatial_token, explicit_channels = self.channel_encoder(channel_tokens)

        temporal = self.temporal_position(self.temporal_input(spatial_token))
        temporal_padding_mask = ~valid_mask
        for block in self.temporal_blocks:
            temporal = block(temporal, key_padding_mask=temporal_padding_mask)
        temporal = self.temporal_norm(temporal)

        pooled, temporal_weights = self.temporal_pool(temporal, valid_mask)
        logits = self.classifier(pooled)

        channel_vote_logits = None
        channel_weights = None
        if self.use_channel_vote:
            channel_scores = self.channel_gate(explicit_channels).squeeze(-1)
            channel_weights = torch.softmax(channel_scores, dim=2)
            per_channel_logits = self.channel_classifier(explicit_channels)
            per_time_vote = torch.sum(
                channel_weights.unsqueeze(-1) * per_channel_logits,
                dim=2,
            )
            time_mask = valid_mask.to(per_time_vote.dtype).unsqueeze(-1)
            channel_vote_logits = (
                (per_time_vote * time_mask).sum(dim=1)
                / time_mask.sum(dim=1).clamp_min(1.0)
            )

        subject_logits = None
        if self.subject_classifier is not None:
            reversed_embedding = GradientReversalFunction.apply(
                pooled,
                float(subject_grl_coefficient),
            )
            subject_logits = self.subject_classifier(reversed_embedding)

        return {
            "logits": logits,
            "channel_vote_logits": channel_vote_logits,
            "subject_logits": subject_logits,
            "embedding": pooled,
            "temporal_weights": temporal_weights,
            "channel_weights": channel_weights,
        }


def build_model(
    dropout: float = DROPOUT,
    use_subject_adversary: bool = USE_SUBJECT_ADVERSARY,
):
    return HierarchicalFBCT(
        channels=len(EEG_CHANNEL_NAMES),
        band_sizes=BAND_SIZES,
        time_steps=30,
        classes=len(EMOTION_NAMES),
        source_subject_classes=len(fit_subjects),
        dropout=dropout,
        use_channel_vote=USE_CHANNEL_VOTE,
        use_subject_adversary=use_subject_adversary,
    )


def parameter_breakdown(model: nn.Module) -> pd.DataFrame:
    groups = {}
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        group = name.split(".", 1)[0]
        groups[group] = groups.get(group, 0) + parameter.numel()
    total = sum(groups.values())
    return pd.DataFrame([
        {
            "module": name,
            "parameters": count,
            "percentage": 100.0 * count / total,
        }
        for name, count in sorted(groups.items(), key=lambda item: item[1], reverse=True)
    ])


model = build_model().to(device)
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

with torch.no_grad(), autocast_context():
    shape_check = model(torch.stack([sample_x, sample_x]).to(device))
assert shape_check["logits"].shape == (2, len(EMOTION_NAMES))

print(f"Trainable parameters: {parameter_count:,}")
print("Output shape:", tuple(shape_check["logits"].shape))
print("Channel-vote auxiliary head:", USE_CHANNEL_VOTE)
print("Subject adversary:", USE_SUBJECT_ADVERSARY)
display(parameter_breakdown(model))

del shape_check, model
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()


Trainable parameters: 15,104,432
Output shape: (2, 9)
Channel-vote auxiliary head: True
Subject adversary: False


,module,parameters,percentage
0,temporal_blocks,12609536,83.482358
1,channel_encoder,1604608,10.623425
2,classifier,531465,3.518603
3,temporal_pool,132609,0.877948
4,temporal_input,131584,0.871162
5,frequency_band_encoder,74652,0.494239
6,temporal_position,15360,0.101692
7,channel_classifier,2825,0.018703
8,temporal_norm,1024,0.006779
9,channel_gate,769,0.005091


### 6. Strict 18-trial overfit gate

This verifies that the hierarchical forward/backward path is capable of memorizing a tiny,
class-balanced subset. The sanity model disables AMP, dropout, label smoothing, channel-vote loss
and subject adversarial loss. Full precision is deliberate: this gate diagnoses
model learnability independently of the production-training AMP choice.


In [7]:
def stratified_indices(labels: np.ndarray, samples_per_class: int) -> np.ndarray:
    chosen = []
    for label in range(len(EMOTION_NAMES)):
        matches = np.flatnonzero(labels == label)
        if len(matches) < samples_per_class:
            raise RuntimeError(f"Class {label} has too few sanity samples")
        chosen.extend(matches[:samples_per_class].tolist())
    return np.asarray(chosen, dtype=np.int64)


def build_adamw(model: nn.Module, learning_rate: float, weight_decay: float):
    kwargs = {
        "lr": learning_rate,
        "weight_decay": weight_decay,
        "betas": (0.9, 0.999),
    }
    if device.type == "cuda" and "fused" in inspect.signature(AdamW).parameters:
        kwargs["fused"] = True
    try:
        return AdamW(model.parameters(), **kwargs)
    except (TypeError, RuntimeError):
        kwargs.pop("fused", None)
        return AdamW(model.parameters(), **kwargs)


sanity_result = {"status": "skipped"}
if RUN_SANITY_GATE:
    seed_everything(SEED)
    sanity_indices_np = stratified_indices(train_labels, SANITY_SAMPLES_PER_CLASS)
    sanity_indices = torch.as_tensor(
        sanity_indices_np,
        dtype=torch.long,
        device=train_split.x.device,
    )
    sanity_x = train_split.x.index_select(0, sanity_indices)
    sanity_y = train_split.y.index_select(0, sanity_indices)

    sanity_model = build_model(
        dropout=0.0,
        use_subject_adversary=False,
    ).to(device)
    sanity_optimizer = build_adamw(
        sanity_model,
        SANITY_LEARNING_RATE,
        weight_decay=0.0,
    )

    initial_loss = None
    final_loss = None
    final_accuracy = 0.0
    completed_steps = 0

    for step in range(1, SANITY_MAX_STEPS + 1):
        sanity_model.train()
        sanity_optimizer.zero_grad(set_to_none=True)
        with contextlib.nullcontext():
            sanity_outputs = sanity_model(sanity_x)
            sanity_loss = nn.functional.cross_entropy(
                sanity_outputs["logits"],
                sanity_y,
            )
        sanity_loss.backward()
        sanity_optimizer.step()

        if initial_loss is None:
            initial_loss = float(sanity_loss.detach().cpu())

        if step == 1 or step % 10 == 0:
            sanity_model.eval()
            with torch.no_grad(), contextlib.nullcontext():
                checked = sanity_model(sanity_x)["logits"]
                checked_loss = nn.functional.cross_entropy(checked, sanity_y)
            final_loss = float(checked_loss.detach().cpu())
            final_accuracy = float(
                (checked.argmax(1) == sanity_y).float().mean().detach().cpu()
            )
            if (
                final_accuracy >= SANITY_TARGET_ACCURACY
                and final_loss <= SANITY_TARGET_LOSS
            ):
                completed_steps = step
                break

    passed = (
        final_accuracy >= SANITY_TARGET_ACCURACY
        and final_loss <= SANITY_TARGET_LOSS
    )
    sanity_result = {
        "status": "passed" if passed else "failed",
        "samples": int(len(sanity_indices_np)),
        "initial_loss": initial_loss,
        "final_loss": final_loss,
        "final_accuracy": final_accuracy,
        "steps": completed_steps or SANITY_MAX_STEPS,
        "dropout": 0.0,
        "weight_decay": 0.0,
        "label_smoothing": 0.0,
        "channel_vote_loss": 0.0,
        "subject_adversarial_loss": 0.0,
        "precision": "float32",
    }
    (RUN_ROOT / "sanity.json").write_text(
        json.dumps(sanity_result, indent=2),
        encoding="utf-8",
    )
    print(json.dumps(sanity_result, indent=2))

    del sanity_model, sanity_optimizer, sanity_x, sanity_y
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    if not passed:
        raise RuntimeError("Hard sanity gate failed; do not interpret full training")
else:
    print("Sanity gate skipped by configuration")


{
  "status": "passed",
  "samples": 18,
  "initial_loss": 2.211543083190918,
  "final_loss": 0.013977368362247944,
  "final_accuracy": 1.0,
  "steps": 40,
  "dropout": 0.0,
  "weight_decay": 0.0,
  "label_smoothing": 0.0,
  "channel_vote_loss": 0.0,
  "subject_adversarial_loss": 0.0,
  "precision": "float32"
}


## Training

### 7. Metrics, scheduling and plotting helpers


In [8]:
@torch.no_grad()
def evaluate_model(model: nn.Module, split: TensorSplit):
    model.eval()
    targets, predictions = [], []
    loss_sum = torch.zeros((), device=device)
    count = 0
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    for value, label, _subject_id, _subject_class in iterate_batches(
        split,
        EVAL_BATCH_SIZE,
        shuffle=False,
    ):
        value = value.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True)
        with autocast_context():
            logits = model(value)["logits"]
            loss = criterion(logits, label)
        loss_sum += loss.detach() * len(label)
        count += len(label)
        targets.append(label.detach().cpu().numpy())
        predictions.append(logits.argmax(1).detach().cpu().numpy())

    y_true = np.concatenate(targets)
    y_pred = np.concatenate(predictions)
    return {
        "loss": float((loss_sum / count).detach().cpu()),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "predicted_classes": int(len(np.unique(y_pred))),
        "prediction_histogram": np.bincount(
            y_pred,
            minlength=len(EMOTION_NAMES),
        ).tolist(),
        "confusion_matrix": confusion_matrix(
            y_true,
            y_pred,
            labels=np.arange(len(EMOTION_NAMES)),
        ).tolist(),
    }


def warmup_cosine_lambda(epoch_index: int) -> float:
    if epoch_index < WARMUP_EPOCHS:
        return float(epoch_index + 1) / max(WARMUP_EPOCHS, 1)
    progress = (
        (epoch_index - WARMUP_EPOCHS)
        / max(EPOCHS - WARMUP_EPOCHS - 1, 1)
    )
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR_RATIO + (1.0 - MIN_LR_RATIO) * cosine


def subject_grl_schedule(epoch: int) -> float:
    if not USE_SUBJECT_ADVERSARY:
        return 0.0
    progress = min(epoch / max(SUBJECT_ADV_WARMUP_EPOCHS, 1), 1.0)
    # Smooth ramp from 0 to 1.
    return float(2.0 / (1.0 + math.exp(-10.0 * progress)) - 1.0)


def plot_confusion(matrix, title, output_path=None):
    values = np.asarray(matrix)
    fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
    image = ax.imshow(values, cmap="Blues")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    ax.set(
        title=title,
        xlabel="Predicted class",
        ylabel="True class",
        xticks=np.arange(len(EMOTION_NAMES)),
        yticks=np.arange(len(EMOTION_NAMES)),
        xticklabels=EMOTION_NAMES,
        yticklabels=EMOTION_NAMES,
    )
    ax.tick_params(axis="x", rotation=45)
    threshold = values.max(initial=0) / 2
    for row in range(len(EMOTION_NAMES)):
        for column in range(len(EMOTION_NAMES)):
            ax.text(
                column,
                row,
                str(values[row, column]),
                ha="center",
                va="center",
                color="white" if values[row, column] > threshold else "black",
                fontsize=8,
            )
    if output_path is not None:
        fig.savefig(output_path, dpi=180)
    plt.show()
    plt.close(fig)


### 8. Train the hierarchical model

The target subjects remain unread. Model selection uses source-development Macro-F1, then
balanced accuracy, then development loss. Evaluation is performed every epoch and early
stopping prevents the model from continuing to memorize the source-fit subjects after
development performance stops improving.


In [9]:
history = []
best_source_dev = None
best_epoch = None
BEST_PATH = RUN_ROOT / "best_source_dev.pt"
FINAL_PATH = RUN_ROOT / "model_final.pt"

if RUN_TRAINING:
    if RUN_SANITY_GATE and sanity_result["status"] != "passed":
        raise RuntimeError("Sanity gate must pass before full training")

    seed_everything(SEED)
    raw_model = build_model().to(device)
    training_model = (
        torch.compile(raw_model)
        if COMPILE_MODEL and hasattr(torch, "compile")
        else raw_model
    )

    optimizer = build_adamw(raw_model, LEARNING_RATE, WEIGHT_DECAY)
    scheduler = LambdaLR(optimizer, lr_lambda=warmup_cosine_lambda)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    scaler_enabled = bool(
        amp_enabled and amp_dtype == torch.float16
    )
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=scaler_enabled)
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=scaler_enabled)

    best_key = (-math.inf, -math.inf, -math.inf)
    epochs_without_improvement = 0
    started = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        training_model.train()
        main_loss_sum = torch.zeros((), device=device)
        total_loss_sum = torch.zeros((), device=device)
        vote_loss_sum = torch.zeros((), device=device)
        subject_loss_sum = torch.zeros((), device=device)
        correct = torch.zeros((), dtype=torch.long, device=device)
        count = 0
        grad_norm_sum = torch.zeros((), device=device)
        steps = 0

        grl_coefficient = subject_grl_schedule(epoch)

        for value, label, _subject_id, subject_class in iterate_batches(
            train_split,
            BATCH_SIZE,
            shuffle=True,
        ):
            value = value.to(device, non_blocking=True)
            label = label.to(device, non_blocking=True)
            subject_class = subject_class.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with autocast_context():
                outputs = training_model(
                    value,
                    subject_grl_coefficient=grl_coefficient,
                )
                main_loss = criterion(outputs["logits"], label)
                total_loss = main_loss

                vote_loss = torch.zeros((), device=device)
                if (
                    USE_CHANNEL_VOTE
                    and outputs["channel_vote_logits"] is not None
                ):
                    vote_loss = criterion(
                        outputs["channel_vote_logits"],
                        label,
                    )
                    total_loss = (
                        total_loss
                        + CHANNEL_VOTE_LOSS_WEIGHT * vote_loss
                    )

                subject_loss = torch.zeros((), device=device)
                if (
                    USE_SUBJECT_ADVERSARY
                    and outputs["subject_logits"] is not None
                ):
                    subject_loss = nn.functional.cross_entropy(
                        outputs["subject_logits"],
                        subject_class,
                    )
                    total_loss = (
                        total_loss
                        + SUBJECT_ADV_WEIGHT * subject_loss
                    )

            if scaler.is_enabled():
                scaler.scale(total_loss).backward()
                scaler.unscale_(optimizer)
                gradient_norm = nn.utils.clip_grad_norm_(
                    raw_model.parameters(),
                    MAX_GRAD_NORM,
                )
                scaler.step(optimizer)
                scaler.update()
            else:
                total_loss.backward()
                gradient_norm = nn.utils.clip_grad_norm_(
                    raw_model.parameters(),
                    MAX_GRAD_NORM,
                )
                optimizer.step()

            batch_size = len(label)
            main_loss_sum += main_loss.detach() * batch_size
            total_loss_sum += total_loss.detach() * batch_size
            vote_loss_sum += vote_loss.detach() * batch_size
            subject_loss_sum += subject_loss.detach() * batch_size
            correct += (outputs["logits"].argmax(1) == label).sum()
            grad_norm_sum += gradient_norm.detach()
            count += batch_size
            steps += 1

        scheduler.step()

        row = {
            "epoch": epoch,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "train_main_loss": float((main_loss_sum / count).detach().cpu()),
            "train_total_loss": float((total_loss_sum / count).detach().cpu()),
            "train_channel_vote_loss": float((vote_loss_sum / count).detach().cpu()),
            "train_subject_loss": float((subject_loss_sum / count).detach().cpu()),
            "train_mode_accuracy": float((correct.float() / count).detach().cpu()),
            "mean_preclip_gradient_norm": float((grad_norm_sum / steps).detach().cpu()),
            "subject_grl_coefficient": grl_coefficient,
            "dev_accuracy": np.nan,
            "dev_balanced_accuracy": np.nan,
            "dev_macro_f1": np.nan,
            "dev_predicted_classes": np.nan,
        }

        improved = False
        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            dev_metrics = evaluate_model(training_model, dev_split)
            row.update({
                "dev_accuracy": dev_metrics["accuracy"],
                "dev_balanced_accuracy": dev_metrics["balanced_accuracy"],
                "dev_macro_f1": dev_metrics["macro_f1"],
                "dev_predicted_classes": dev_metrics["predicted_classes"],
            })
            key = (
                dev_metrics["macro_f1"],
                dev_metrics["balanced_accuracy"],
                -dev_metrics["loss"],
            )
            if key > best_key:
                improved = True
                best_key = key
                best_epoch = epoch
                best_source_dev = dev_metrics
                epochs_without_improvement = 0
                torch.save({
                    "model_state_dict": {
                        name: tensor.detach().cpu()
                        for name, tensor in raw_model.state_dict().items()
                    },
                    "feature_mean": feature_mean,
                    "feature_std": feature_std,
                    "reference": reference,
                    "best_epoch": best_epoch,
                    "source_dev_metrics": best_source_dev,
                    "model_preset": MODEL_PRESET,
                    "model_config": MODEL_CFG,
                    "use_channel_vote": USE_CHANNEL_VOTE,
                    "use_subject_adversary": USE_SUBJECT_ADVERSARY,
                    "target_loaded_during_selection": False,
                }, BEST_PATH)
            else:
                epochs_without_improvement += 1

            print(
                f"epoch {epoch:03d}/{EPOCHS} "
                f"main={row['train_main_loss']:.4f} "
                f"total={row['train_total_loss']:.4f} "
                f"train={row['train_mode_accuracy']:.3f} "
                f"dev_bacc={dev_metrics['balanced_accuracy']:.3f} "
                f"dev_f1={dev_metrics['macro_f1']:.3f} "
                f"classes={dev_metrics['predicted_classes']} "
                f"{'*' if improved else ''}"
            )
        else:
            print(
                f"epoch {epoch:03d}/{EPOCHS} "
                f"main={row['train_main_loss']:.4f} "
                f"train={row['train_mode_accuracy']:.3f}"
            )

        history.append(row)

        if (
            EARLY_STOPPING_PATIENCE > 0
            and epochs_without_improvement >= EARLY_STOPPING_PATIENCE
        ):
            print(
                f"Early stopping at epoch {epoch}; "
                f"best source-development epoch was {best_epoch}."
            )
            break

    completed_epochs = len(history)
    final_source_train = evaluate_model(training_model, train_split)
    final_source_dev = evaluate_model(training_model, dev_split)

    torch.save({
        "model_state_dict": {
            name: tensor.detach().cpu()
            for name, tensor in raw_model.state_dict().items()
        },
        "feature_mean": feature_mean,
        "feature_std": feature_std,
        "reference": reference,
        "epoch": completed_epochs,
        "source_train_metrics": final_source_train,
        "source_dev_metrics": final_source_dev,
        "model_preset": MODEL_PRESET,
        "model_config": MODEL_CFG,
        "use_channel_vote": USE_CHANNEL_VOTE,
        "use_subject_adversary": USE_SUBJECT_ADVERSARY,
        "target_loaded_during_training": False,
    }, FINAL_PATH)

    pd.DataFrame(history).to_csv(
        RUN_ROOT / "training_history.csv",
        index=False,
    )
    summary = {
        "status": "source_training_complete",
        "model": "hierarchical Frequency-Band-Channel-Time Transformer",
        "model_preset": MODEL_PRESET,
        "model_config": MODEL_CFG,
        "parameter_count": parameter_count,
        "feature": "frequency_resolved_sqrt_jsd_46_native_bins",
        "band_sizes": list(BAND_SIZES),
        "channel_vote": USE_CHANNEL_VOTE,
        "channel_vote_loss_weight": CHANNEL_VOTE_LOSS_WEIGHT,
        "subject_adversary": USE_SUBJECT_ADVERSARY,
        "subject_adversary_weight": SUBJECT_ADV_WEIGHT,
        "batch_size": BATCH_SIZE,
        "amp": amp_enabled,
        "amp_dtype": AMP_DTYPE,
        "cache_data_on_device": CACHE_DATA_ON_DEVICE,
        "completed_epochs": completed_epochs,
        "best_source_dev_epoch": best_epoch,
        "best_source_dev": best_source_dev,
        "final_source_train": final_source_train,
        "final_source_dev": final_source_dev,
        "target_loaded": False,
        "elapsed_seconds": time.perf_counter() - started,
    }
    (RUN_ROOT / "source_training_summary.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print(json.dumps(summary, indent=2, ensure_ascii=False))
else:
    print("RUN_TRAINING=False: model construction and data-path validation only")


epoch 001/80 main=2.2183 total=2.5489 train=0.122 dev_bacc=0.111 dev_f1=0.022 classes=2 *
epoch 002/80 main=2.2183 total=2.5501 train=0.124 dev_bacc=0.111 dev_f1=0.028 classes=1 *
epoch 003/80 main=2.2097 total=2.5434 train=0.129 dev_bacc=0.111 dev_f1=0.041 classes=2 *
epoch 004/80 main=2.2019 total=2.5336 train=0.125 dev_bacc=0.111 dev_f1=0.028 classes=1 
epoch 005/80 main=2.2064 total=2.5396 train=0.120 dev_bacc=0.111 dev_f1=0.022 classes=1 
epoch 006/80 main=2.2030 total=2.5345 train=0.127 dev_bacc=0.111 dev_f1=0.028 classes=1 
epoch 007/80 main=2.2010 total=2.5322 train=0.139 dev_bacc=0.111 dev_f1=0.028 classes=1 
epoch 008/80 main=2.2022 total=2.5337 train=0.136 dev_bacc=0.111 dev_f1=0.028 classes=1 
epoch 009/80 main=2.1970 total=2.5273 train=0.144 dev_bacc=0.111 dev_f1=0.022 classes=2 
epoch 010/80 main=2.1981 total=2.5282 train=0.132 dev_bacc=0.111 dev_f1=0.028 classes=1 
epoch 011/80 main=2.1973 total=2.5264 train=0.141 dev_bacc=0.111 dev_f1=0.028 classes=1 
epoch 012/80 main=

## Results

### 9. Source-only curves and selected confusion matrix


In [10]:
if history:
    history_frame = pd.DataFrame(history)
    evaluated = history_frame.dropna(subset=["dev_macro_f1"])

    fig, axes = plt.subplots(1, 3, figsize=(17, 4), constrained_layout=True)
    axes[0].plot(
        history_frame["epoch"],
        history_frame["train_main_loss"],
        label="main CE",
    )
    axes[0].plot(
        history_frame["epoch"],
        history_frame["train_total_loss"],
        label="total",
        alpha=0.8,
    )
    axes[0].set(
        title="Training losses",
        xlabel="Epoch",
        ylabel="Loss",
    )
    axes[0].legend()

    axes[1].plot(
        history_frame["epoch"],
        history_frame["train_mode_accuracy"],
    )
    axes[1].set(
        title="Training accuracy",
        xlabel="Epoch",
        ylabel="Accuracy",
    )

    axes[2].plot(
        evaluated["epoch"],
        evaluated["dev_balanced_accuracy"],
        marker="o",
        label="BAcc",
    )
    axes[2].plot(
        evaluated["epoch"],
        evaluated["dev_macro_f1"],
        marker="o",
        label="Macro-F1",
    )
    axes[2].axhline(
        1 / len(EMOTION_NAMES),
        color="gray",
        linestyle="--",
        label="chance BAcc",
    )
    axes[2].set(
        title="Source-development metrics",
        xlabel="Epoch",
        ylabel="Score",
    )
    axes[2].legend()

    fig.savefig(RUN_ROOT / "source_training_curves.png", dpi=180)
    plt.show()
    plt.close(fig)

    if best_source_dev is not None:
        plot_confusion(
            best_source_dev["confusion_matrix"],
            f"Best source-development confusion (epoch {best_epoch})",
            RUN_ROOT / "best_source_dev_confusion.png",
        )

    display(evaluated[[
        "epoch",
        "train_mode_accuracy",
        "dev_accuracy",
        "dev_balanced_accuracy",
        "dev_macro_f1",
        "dev_predicted_classes",
    ]])
else:
    print("No training history yet")


C:\Users\Lin\AppData\Local\Temp\ipykernel_49968\1970361010.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Lin\AppData\Local\Temp\ipykernel_49968\3917567396.py:92: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,epoch,train_mode_accuracy,dev_accuracy,dev_balanced_accuracy,dev_macro_f1,dev_predicted_classes
0,1,0.122104,0.107143,0.111111,0.021524,2
1,2,0.124035,0.142857,0.111111,0.027778,1
2,3,0.129344,0.116795,0.110611,0.041433,2
3,4,0.125000,0.142857,0.111111,0.027778,1
4,5,0.120174,0.107143,0.111111,0.021505,1
5,6,0.126931,0.142857,0.111111,0.027778,1
6,7,0.139479,0.142857,0.111111,0.027778,1
7,8,0.135618,0.142857,0.111111,0.027778,1
8,9,0.143822,0.107143,0.111111,0.021524,2
9,10,0.131757,0.142857,0.111111,0.027778,1


## Optional locked target evaluation

This cell remains disabled by default. Only enable it after the architecture, training
configuration and source-development checkpoint rule are frozen. Do not tune the same
protocol after observing the outer-target result.


In [11]:
if not EVALUATE_TARGET_AFTER_LOCK:
    print("Outer target remains locked and unread.")
else:
    checkpoint_path = (
        BEST_PATH
        if TARGET_CHECKPOINT == "best_source_dev"
        else FINAL_PATH
    )
    if not checkpoint_path.is_file():
        raise FileNotFoundError(
            f"Train and lock the source checkpoint first: {checkpoint_path}"
        )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )
    locked_reference = checkpoint["reference"]
    locked_mean = np.asarray(checkpoint["feature_mean"], dtype=np.float32)
    locked_std = np.asarray(checkpoint["feature_std"], dtype=np.float32)

    target_raw = materialize_split(
        store,
        outer_target_subjects,
        locked_reference,
        EPSILON,
        FEATURE_STORAGE_DTYPE,
    )
    target_features = compact_native_bins(target_raw.pop("x"))
    target_labels = np.asarray(target_raw["y"], dtype=np.int64)
    target_subject_ids = np.asarray(target_raw["subjects"], dtype=np.int64)
    target_error = float(target_raw["maximum_invariant_error"])

    target_split = make_tensor_split(
        target_features,
        target_labels,
        target_subject_ids,
        locked_mean,
        locked_std,
        None,
    )
    del target_features, target_raw
    gc.collect()

    locked_model = build_model().to(device)
    locked_model.load_state_dict(checkpoint["model_state_dict"])
    target_metrics = evaluate_model(locked_model, target_split)

    target_result = {
        "status": "outer_target_evaluated_after_source_lock",
        "checkpoint": TARGET_CHECKPOINT,
        "metrics": target_metrics,
        "target_subjects": list(outer_target_subjects),
        "target_used_for_selection": False,
        "post_target_tuning_permitted": False,
        "maximum_sqrt_jsd_reconstruction_error": target_error,
    }
    (RUN_ROOT / "locked_target_result.json").write_text(
        json.dumps(target_result, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print(json.dumps(target_result, indent=2, ensure_ascii=False))
    plot_confusion(
        target_metrics["confusion_matrix"],
        "Locked outer-target confusion matrix",
        RUN_ROOT / "locked_target_confusion.png",
    )


Outer target remains locked and unread.


## Checks and interpretation boundary

- `source_isolation_audit.json` must report `target_loaded=false`.
- The sanity gate must pass before low full-training metrics are interpreted.
- Compare the default architecture-only run against the legacy notebook before enabling
  subject adversarial learning.
- A high source-fit score with low source-development score is still cross-subject
  overfitting; increasing the preset size is not the first response.
- If the `base` preset is stable and GPU memory remains available, switch to `large` and use
  a new `RUN_NAME`.
- If GPU utilization is still low, first increase `BATCH_SIZE` until memory is efficiently
  occupied. Do not increase every model dimension simultaneously.
- The current input remains non-directional `sqrt-JSD`. A later feature ablation should compare
  it with signed log-PSD ratio and `DE + JSD`, without changing this architecture.
- This notebook is a single outer-fold architecture experiment, not the final multi-fold
  paper result.
